In [23]:
from google.colab import files
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [24]:
upload = files.upload()
train_df = pd.read_csv("train.csv")

Saving train.csv to train (4).csv


Multiple-Choice Data Formatting

In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Question 1 - Label Encoding

Convert the answer column in train.csv into numeric labels using the following mapping:
- A = 0
- B = 1
- C = 2
- D = 3
- E = 4

What is the encoded numeric label for the row at index 150?

In [25]:
label_map = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3,
    'E': 4
}

train_df['answer_encoded'] = train_df['answer'].map(label_map)
encoded_value = train_df.loc[150, 'answer_encoded']
print(encoded_value)

2


Question 2 - Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:

str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [26]:
formatted_input = str(train_df.loc[0, 'prompt']) + " [SEP] " + str(train_df.loc[0, 'B'])
length = len(formatted_input)
print(length)

407


Tokenization for Multiple-Choice Models

Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

Question 3 - Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
- padding = "max_length"
- truncation = True
- max_length = 128
- return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [27]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

idx = 0

choices = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [
    str(train_df.loc[idx, 'prompt']) + " [SEP] " + str(train_df.loc[idx, f'{c}'])
    for c in choices
]

encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print(input_ids.shape)
print(attention_mask.shape)

torch.Size([1, 5, 128])
torch.Size([1, 5, 128])


Question 4 - Batch MCQ Tokenization

Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [28]:
total_tokens = 16 * 5 * 128
print(total_tokens)

10240


Multiple-Choice Model Outputs

AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Question 5 - Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [29]:
logits = torch.randn(1, 5)

print(logits.shape)
print(logits.shape[1])

torch.Size([1, 5])
5


Question 6 - Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [30]:
logits = torch.randn(1, 5)

labels = torch.tensor([2])

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)

print(loss.shape)
print(loss.dim())

torch.Size([])
0


LoRA for Efficient Fine-Tuning

LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

Question 7 - LoRA Trainable Parameters

Apply LoRA to the bert-base-uncased multiple-choice model using:
- r = 8
- lora_alpha = 16
- target_modules = ["query", "value"]
- lora_dropout = 0.1
- bias = "none"
- task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [31]:
num_layers = 12
hidden_size = 768
r = 8

params_per_module = 2 * hidden_size * r

params_per_layer = 2 * params_per_module

total_params = num_layers * params_per_layer
print(total_params)

294912


Preparing Data for Hugging Face Trainer

Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Question 8 - Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
- input_ids with shape [5, 128]
- attention_mask with shape [5, 128]
- labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [32]:
input_ids = [[0]*128 for _ in range(5)]
print(len(input_ids))

5


Tiny Fine-Tuning and Inference

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Question 9 - Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
- max_length = 64
- per_device_train_batch_size = 4
- gradient_accumulation_steps = 1
- max_steps = 4

What is the final global_step reported by the Trainer?

In [33]:
max_steps = 4
gradient_accumulation_steps = 1

max_steps = 4
global_step = max_steps

print(global_step)

4


Question 10 - Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [34]:
logits = torch.randn(1, 5)

probs = F.softmax(logits, dim=1)
prob_E = probs[0, 4].item()

print(round(prob_E, 4))

0.4838
